In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime

# ============================================================
# 1. CONFIG
# ============================================================
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

N_PARENTS = 5
CHILD_PER_PARENT = 3

SIZE_LIST = ["S", "M", "L"]

PRICE_MIN = 20
PRICE_MAX = 100

LIFECYCLE_DISTRIBUTION = {
    "Evergreen": 0.40,
    "Trend": 0.25,
    "Seasonal": 0.20,
    "Declining": 0.15
}

STORES = [
    {
        "store_id": "S001",
        "store_name": "Downtown Store",
        "store_cluster": "Downtown",
        "store_tier": "A",
        "traffic_level": "High"
    },
    {
        "store_id": "S002",
        "store_name": "Urban Store",
        "store_cluster": "Urban",
        "store_tier": "B",
        "traffic_level": "Medium"
    }
]

CATEGORY = "Apparel"
BRAND = "SimuBrand"

# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================
def assign_lifecycle_list(n_items, lifecycle_distribution):
    lifecycle_names = list(lifecycle_distribution.keys())
    lifecycle_probs = list(lifecycle_distribution.values())

    assigned = np.random.choice(
        lifecycle_names,
        size=n_items,
        p=lifecycle_probs
    )
    return assigned.tolist()


def generate_parent_sku_code(parent_num):
    return f"P{str(parent_num).zfill(3)}"


def generate_child_sku_code(parent_code, size_code):
    return f"{parent_code}_{size_code}"


def generate_product_name(parent_num):
    product_names = [
        "Basic Tee",
        "Relaxed Shirt",
        "Summer Dress",
        "Classic Polo",
        "Wide Leg Pants",
        "Cropped Top",
        "Casual Hoodie",
        "Linen Shirt",
        "Midi Skirt",
        "Straight Jeans"
    ]
    base_name = product_names[(parent_num - 1) % len(product_names)]
    return f"{base_name} {parent_num}"


def random_price():
    return random.randint(PRICE_MIN, PRICE_MAX)


def random_age_weeks():
    return random.randint(4, 104)


# ============================================================
# 3. STORE MASTER
# ============================================================
store_master = pd.DataFrame(STORES)

# ============================================================
# 4. PARENT PRODUCT MASTER
# ============================================================
parent_lifecycle_list = assign_lifecycle_list(
    n_items=N_PARENTS,
    lifecycle_distribution=LIFECYCLE_DISTRIBUTION
)

parent_rows = []

for i in range(1, N_PARENTS + 1):
    parent_code = generate_parent_sku_code(i)
    product_name = generate_product_name(i)
    base_price = random_price()
    lifecycle = parent_lifecycle_list[i - 1]
    sku_age_weeks = random_age_weeks()

    parent_rows.append({
        "parent_sku_id": parent_code,
        "product_name": product_name,
        "brand": BRAND,
        "category": CATEGORY,
        "lifecycle_type": lifecycle,
        "base_price": base_price,
        "sku_age_weeks": sku_age_weeks,
        "size_variant_count": CHILD_PER_PARENT,
        "is_shared_assortment": 1
    })

product_master_parent = pd.DataFrame(parent_rows)

# ============================================================
# 5. CHILD PRODUCT MASTER
# ============================================================
child_rows = []

for _, parent_row in product_master_parent.iterrows():
    parent_sku_id = parent_row["parent_sku_id"]
    product_name = parent_row["product_name"]
    base_price = parent_row["base_price"]
    lifecycle_type = parent_row["lifecycle_type"]
    sku_age_weeks = parent_row["sku_age_weeks"]

    for size_code in SIZE_LIST:
        child_sku_id = generate_child_sku_code(parent_sku_id, size_code)

        child_rows.append({
            "child_sku_id": child_sku_id,
            "parent_sku_id": parent_sku_id,
            "product_name": product_name,
            "brand": BRAND,
            "category": CATEGORY,
            "size": size_code,
            "base_price": base_price,
            "lifecycle_type": lifecycle_type,
            "sku_age_weeks": sku_age_weeks,
            "variant_type": "Size"
        })

product_master_child = pd.DataFrame(child_rows)

# ============================================================
# 6. OPTIONAL: STORE-SKU MATRIX
#    Shared assortment means every child SKU exists in both stores
# ============================================================
store_sku_rows = []

for _, store_row in store_master.iterrows():
    for _, child_row in product_master_child.iterrows():
        store_sku_rows.append({
            "store_id": store_row["store_id"],
            "store_name": store_row["store_name"],
            "child_sku_id": child_row["child_sku_id"],
            "parent_sku_id": child_row["parent_sku_id"],
            "base_price": child_row["base_price"],
            "lifecycle_type": child_row["lifecycle_type"],
            "sku_age_weeks": child_row["sku_age_weeks"]
        })

store_sku_master = pd.DataFrame(store_sku_rows)

# ============================================================
# 7. QUICK CHECKS
# ============================================================
print("Store Master")
display(store_master)

print("Parent Product Master")
display(product_master_parent)

print("Child Product Master")
display(product_master_child)

print("Store-SKU Master")
display(store_sku_master)

print("Lifecycle Distribution in Parent Master")
display(product_master_parent["lifecycle_type"].value_counts(dropna=False))

print("Parent Count:", len(product_master_parent))
print("Child Count:", len(product_master_child))
print("Store-SKU Count:", len(store_sku_master))

Store Master


,store_id,store_name,store_cluster,store_tier,traffic_level
0,S001,Downtown Store,Downtown,A,High
1,S002,Urban Store,Urban,B,Medium


Parent Product Master


,parent_sku_id,product_name,brand,category,lifecycle_type,base_price,sku_age_weeks,size_variant_count,is_shared_assortment
0,P001,Basic Tee 1,SimuBrand,Apparel,Evergreen,34,7,3,1
1,P002,Relaxed Shirt 2,SimuBrand,Apparel,Declining,55,35,3,1
2,P003,Summer Dress 3,SimuBrand,Apparel,Seasonal,48,21,3,1
3,P004,Classic Polo 4,SimuBrand,Apparel,Trend,33,90,3,1
4,P005,Wide Leg Pants 5,SimuBrand,Apparel,Evergreen,89,15,3,1


Child Product Master


...

Store-SKU Master


,store_id,store_name,child_sku_id,parent_sku_id,base_price,lifecycle_type,sku_age_weeks
0,S001,Downtown Store,P001_S,P001,34,Evergreen,7
1,S001,Downtown Store,P001_M,P001,34,Evergreen,7
2,S001,Downtown Store,P001_L,P001,34,Evergreen,7
3,S001,Downtown Store,P002_S,P002,55,Declining,35
4,S001,Downtown Store,P002_M,P002,55,Declining,35
5,S001,Downtown Store,P002_L,P002,55,Declining,35
6,S001,Downtown Store,P003_S,P003,48,Seasonal,21
7,S001,Downtown Store,P003_M,P003,48,Seasonal,21
8,S001,Downtown Store,P003_L,P003,48,Seasonal,21
9,S001,Downtown Store,P004_S,P004,33,Trend,90


Lifecycle Distribution in Parent Master


lifecycle_type
Evergreen    2
Declining    1
Seasonal     1
Trend        1
Name: count, dtype: int64

Parent Count: 5
Child Count: 15
Store-SKU Count: 30


In [2]:
# ============================================================
# 8. WEEKLY SALES FACT TABLE GENERATION
# ============================================================
from datetime import timedelta

N_WEEKS = 26
START_WEEK = "2025-01-06"   # Monday anchor week

STORE_DEMAND_MULTIPLIER = {
    "S001": 1.20,   # Downtown
    "S002": 0.85    # Urban
}

LIFECYCLE_DEMAND_MULTIPLIER = {
    "Evergreen": 1.00,
    "Trend": 1.20,
    "Seasonal": 0.95,
    "Declining": 0.70
}

# Number of special weeks across the whole simulation horizon
N_SPIKE_WEEKS = 4
N_SLOW_WEEKS = 4

# In normal weeks, at most 10% of total child-store combinations have sales
NORMAL_WEEK_ACTIVE_RATE = 0.10

# In spike weeks, there should be 5 to 10 child-store combinations with sales
SPIKE_MIN_ACTIVE = 5
SPIKE_MAX_ACTIVE = 10

# Qty rules
NORMAL_QTY_MIN = 1
NORMAL_QTY_MAX = 3

SPIKE_QTY_MIN = 1
SPIKE_QTY_MAX = 5


# ============================================================
# 9. BUILD WEEK CALENDAR
# ============================================================
week_start_dates = pd.date_range(start=START_WEEK, periods=N_WEEKS, freq="W-MON")

week_df = pd.DataFrame({
    "week_start_date": week_start_dates
})

week_df["week_id"] = ["W" + str(i + 1).zfill(3) for i in range(len(week_df))]
week_df["week_end_date"] = week_df["week_start_date"] + pd.Timedelta(days=6)


# ============================================================
# 10. ASSIGN WEEK TYPE
#     - spike weeks
#     - slow weeks
#     - normal weeks
# ============================================================
all_week_indices = list(range(N_WEEKS))

spike_week_indices = sorted(random.sample(all_week_indices, N_SPIKE_WEEKS))
remaining_indices = [x for x in all_week_indices if x not in spike_week_indices]
slow_week_indices = sorted(random.sample(remaining_indices, N_SLOW_WEEKS))

week_type_list = []

for i in range(N_WEEKS):
    if i in spike_week_indices:
        week_type_list.append("spike")
    elif i in slow_week_indices:
        week_type_list.append("slow")
    else:
        week_type_list.append("normal")

week_df["week_type"] = week_type_list


# ============================================================
# 11. PREP BASE STORE-SKU UNIVERSE
# ============================================================
sales_base = store_sku_master.copy()

# Optional helper lookup for store multiplier
sales_base["store_multiplier"] = sales_base["store_id"].map(STORE_DEMAND_MULTIPLIER)

# Optional helper lookup for lifecycle multiplier
sales_base["lifecycle_multiplier"] = sales_base["lifecycle_type"].map(LIFECYCLE_DEMAND_MULTIPLIER)

total_store_child_combinations = len(sales_base)


# ============================================================
# 12. GENERATE WEEKLY SALES
# ============================================================
weekly_sales_rows = []

for _, week_row in week_df.iterrows():
    week_id = week_row["week_id"]
    week_start_date = week_row["week_start_date"]
    week_end_date = week_row["week_end_date"]
    week_type = week_row["week_type"]

    # --------------------------------------------
    # SLOW WEEK => completely zero sales
    # --------------------------------------------
    if week_type == "slow":
        for _, sku_row in sales_base.iterrows():
            weekly_sales_rows.append({
                "week_id": week_id,
                "week_start_date": week_start_date,
                "week_end_date": week_end_date,
                "week_type": week_type,
                "store_id": sku_row["store_id"],
                "store_name": sku_row["store_name"],
                "parent_sku_id": sku_row["parent_sku_id"],
                "child_sku_id": sku_row["child_sku_id"],
                "lifecycle_type": sku_row["lifecycle_type"],
                "base_price": sku_row["base_price"],
                "sales_qty": 0,
                "sales_value": 0
            })

    # --------------------------------------------
    # SPIKE WEEK => 5 to 10 child-store combos active
    # --------------------------------------------
    elif week_type == "spike":
        active_count = random.randint(SPIKE_MIN_ACTIVE, SPIKE_MAX_ACTIVE)

        candidate_df = sales_base.copy()
        candidate_df["selection_weight"] = (
            candidate_df["store_multiplier"] *
            candidate_df["lifecycle_multiplier"]
        )

        sampled_indices = np.random.choice(
            candidate_df.index,
            size=active_count,
            replace=False,
            p=candidate_df["selection_weight"] / candidate_df["selection_weight"].sum()
        )

        active_index_set = set(sampled_indices)

        for idx, sku_row in candidate_df.iterrows():
            if idx in active_index_set:
                qty = random.randint(SPIKE_QTY_MIN, SPIKE_QTY_MAX)
            else:
                qty = 0

            sales_value = qty * sku_row["base_price"]

            weekly_sales_rows.append({
                "week_id": week_id,
                "week_start_date": week_start_date,
                "week_end_date": week_end_date,
                "week_type": week_type,
                "store_id": sku_row["store_id"],
                "store_name": sku_row["store_name"],
                "parent_sku_id": sku_row["parent_sku_id"],
                "child_sku_id": sku_row["child_sku_id"],
                "lifecycle_type": sku_row["lifecycle_type"],
                "base_price": sku_row["base_price"],
                "sales_qty": qty,
                "sales_value": sales_value
            })

    # --------------------------------------------
    # NORMAL WEEK => no more than 10% active
    # --------------------------------------------
    else:
        max_active_count = max(1, int(total_store_child_combinations * NORMAL_WEEK_ACTIVE_RATE))
        active_count = random.randint(0, max_active_count)

        candidate_df = sales_base.copy()
        candidate_df["selection_weight"] = (
            candidate_df["store_multiplier"] *
            candidate_df["lifecycle_multiplier"]
        )

        if active_count > 0:
            sampled_indices = np.random.choice(
                candidate_df.index,
                size=active_count,
                replace=False,
                p=candidate_df["selection_weight"] / candidate_df["selection_weight"].sum()
            )
            active_index_set = set(sampled_indices)
        else:
            active_index_set = set()

        for idx, sku_row in candidate_df.iterrows():
            if idx in active_index_set:
                qty = random.randint(NORMAL_QTY_MIN, NORMAL_QTY_MAX)
            else:
                qty = 0

            sales_value = qty * sku_row["base_price"]

            weekly_sales_rows.append({
                "week_id": week_id,
                "week_start_date": week_start_date,
                "week_end_date": week_end_date,
                "week_type": week_type,
                "store_id": sku_row["store_id"],
                "store_name": sku_row["store_name"],
                "parent_sku_id": sku_row["parent_sku_id"],
                "child_sku_id": sku_row["child_sku_id"],
                "lifecycle_type": sku_row["lifecycle_type"],
                "base_price": sku_row["base_price"],
                "sales_qty": qty,
                "sales_value": sales_value
            })

weekly_sales_fact = pd.DataFrame(weekly_sales_rows)


# ============================================================
# 13. OPTIONAL DERIVED FIELDS
# ============================================================
weekly_sales_fact["has_sales"] = np.where(weekly_sales_fact["sales_qty"] > 0, 1, 0)


# ============================================================
# 14. QUICK CHECKS
# ============================================================
print("Week Calendar")
display(week_df)

print("Weekly Sales Fact Sample")
display(weekly_sales_fact.head(20))

print("Shape of weekly_sales_fact:", weekly_sales_fact.shape)

print("Sales summary by week type")
display(
    weekly_sales_fact.groupby("week_type")[["sales_qty", "sales_value"]]
    .sum()
    .reset_index()
)

print("Active store-child combinations by week")
weekly_active_check = (
    weekly_sales_fact.groupby(["week_id", "week_type"])["has_sales"]
    .sum()
    .reset_index()
    .rename(columns={"has_sales": "active_store_child_count"})
)
display(weekly_active_check)

print("Store level sales summary")
display(
    weekly_sales_fact.groupby(["store_id", "store_name"])[["sales_qty", "sales_value"]]
    .sum()
    .reset_index()
)

Week Calendar


,week_start_date,week_id,week_end_date,week_type
0,2025-01-06,W001,2025-01-12,spike
1,2025-01-13,W002,2025-01-19,spike
2,2025-01-20,W003,2025-01-26,normal
3,2025-01-27,W004,2025-02-02,normal
4,2025-02-03,W005,2025-02-09,slow
5,2025-02-10,W006,2025-02-16,normal
6,2025-02-17,W007,2025-02-23,normal
7,2025-02-24,W008,2025-03-02,normal
8,2025-03-03,W009,2025-03-09,slow
9,2025-03-10,W010,2025-03-16,slow


Weekly Sales Fact Sample


,week_id,week_start_date,week_end_date,week_type,store_id,store_name,parent_sku_id,child_sku_id,lifecycle_type,base_price,sales_qty,sales_value,has_sales
0,W001,2025-01-06,2025-01-12,spike,S001,Downtown Store,P001,P001_S,Evergreen,34,1,34,1
1,W001,2025-01-06,2025-01-12,spike,S001,Downtown Store,P001,P001_M,Evergreen,34,5,170,1
2,W001,2025-01-06,2025-01-12,spike,S001,Downtown Store,P001,P001_L,Evergreen,34,0,0,0
3,W001,2025-01-06,2025-01-12,spike,S001,Downtown Store,P002,P002_S,Declining,55,0,0,0
4,W001,2025-01-06,2025-01-12,spike,S001,Downtown Store,P002,P002_M,Declining,55,2,110,1
5,W001,2025-01-06,2025-01-12,spike,S001,Downtown Store,P002,P002_L,Declining,55,0,0,0
6,W001,2025-01-06,2025-01-12,spike,S001,Downtown Store,P003,P003_S,Seasonal,48,5,240,1
7,W001,2025-01-06,2025-01-12,spike,S001,Downtown Store,P003,P003_M,Seasonal,48,0,0,0
8,W001,2025-01-06,2025-01-12,spike,S001,Downtown Store,P003,P003_L,Seasonal,48,0,0,0
9,W001,2025-01-06,2025-01-12,spike,S001,Downtown Store,P004,P004_S,Trend,33,0,0,0


Shape of weekly_sales_fact: (780, 13)
Sales summary by week type


,week_type,sales_qty,sales_value
0,normal,55,2561
1,slow,0,0
2,spike,81,4289


Active store-child combinations by week


,week_id,week_type,active_store_child_count
0,W001,spike,9
1,W002,spike,5
2,W003,normal,1
3,W004,normal,0
4,W005,slow,0
5,W006,normal,0
6,W007,normal,3
7,W008,normal,2
8,W009,slow,0
9,W010,slow,0


Store level sales summary


,store_id,store_name,sales_qty,sales_value
0,S001,Downtown Store,78,3938
1,S002,Urban Store,58,2912


In [3]:
# ============================================================
# 15. REPLENISHMENT SUPPORT TABLE
# ============================================================

# Assumption:
# We build replenishment support based on weekly_sales_fact
# at store_id + child_sku_id + week_id grain

# -----------------------------
# Config
# -----------------------------
LEAD_TIME_BY_STORE = {
    "S001": 1,   # Downtown
    "S002": 2    # Urban
}

LIFECYCLE_SAFETY_MULTIPLIER = {
    "Evergreen": 1.20,
    "Trend": 1.50,
    "Seasonal": 1.30,
    "Declining": 0.80
}

MIN_SAFETY_STOCK = 1

# -----------------------------
# Sort for rolling calculation
# -----------------------------
replen_df = weekly_sales_fact.copy()

replen_df = replen_df.sort_values(
    by=["store_id", "child_sku_id", "week_start_date"]
).reset_index(drop=True)

# -----------------------------
# Add lead time
# -----------------------------
replen_df["lead_time_weeks"] = replen_df["store_id"].map(LEAD_TIME_BY_STORE)

# -----------------------------
# Last 4 weeks avg sales
# Exclude current week from the rolling avg
# -----------------------------
replen_df["last_4w_avg_sales"] = (
    replen_df.groupby(["store_id", "child_sku_id"])["sales_qty"]
    .transform(lambda s: s.shift(1).rolling(window=4, min_periods=1).mean())
)

replen_df["last_4w_avg_sales"] = replen_df["last_4w_avg_sales"].fillna(0)

# -----------------------------
# Add lifecycle safety multiplier
# -----------------------------
replen_df["safety_multiplier"] = replen_df["lifecycle_type"].map(LIFECYCLE_SAFETY_MULTIPLIER)

# -----------------------------
# Safety stock
# Simple rule:
# safety_stock = ceil(last_4w_avg_sales * safety_multiplier)
# with a minimum floor
# -----------------------------
replen_df["safety_stock_qty"] = np.ceil(
    replen_df["last_4w_avg_sales"] * replen_df["safety_multiplier"]
).astype(int)

replen_df["safety_stock_qty"] = replen_df["safety_stock_qty"].clip(lower=MIN_SAFETY_STOCK)

# -----------------------------
# Target stock
# Simple rule:
# target stock = demand during lead time + safety stock
# -----------------------------
replen_df["target_stock_qty"] = np.ceil(
    (replen_df["last_4w_avg_sales"] * replen_df["lead_time_weeks"]) +
    replen_df["safety_stock_qty"]
).astype(int)

# -----------------------------
# Simulate on-hand stock
# We want stock to be partly related to demand
# but still have under / over / healthy cases
# -----------------------------
def simulate_on_hand_qty(row):
    target_stock = row["target_stock_qty"]

    # random stock factor to create mixed inventory states
    stock_factor = random.uniform(0.4, 1.6)

    simulated_stock = int(round(target_stock * stock_factor))

    if simulated_stock < 0:
        simulated_stock = 0

    return simulated_stock

replen_df["on_hand_qty"] = replen_df.apply(simulate_on_hand_qty, axis=1)

# -----------------------------
# Reorder qty
# -----------------------------
replen_df["reorder_qty"] = replen_df["target_stock_qty"] - replen_df["on_hand_qty"]
replen_df["reorder_qty"] = replen_df["reorder_qty"].clip(lower=0)

# -----------------------------
# Weeks of cover
# -----------------------------
def calculate_weeks_of_cover(row):
    avg_sales = row["last_4w_avg_sales"]
    on_hand = row["on_hand_qty"]

    if avg_sales <= 0:
        return np.nan

    return round(on_hand / avg_sales, 2)

replen_df["weeks_of_cover"] = replen_df.apply(calculate_weeks_of_cover, axis=1)

# -----------------------------
# Stock status
# -----------------------------
def assign_stock_status(row):
    on_hand = row["on_hand_qty"]
    safety_stock = row["safety_stock_qty"]
    target_stock = row["target_stock_qty"]

    if on_hand <= safety_stock:
        return "Understock"
    elif on_hand > target_stock * 1.25:
        return "Overstock"
    else:
        return "Healthy"

replen_df["stock_status"] = replen_df.apply(assign_stock_status, axis=1)

# -----------------------------
# Priority flag
# -----------------------------
def assign_priority_flag(row):
    if row["stock_status"] == "Understock" and row["last_4w_avg_sales"] > 0:
        return "Reorder Now"
    elif row["stock_status"] == "Overstock":
        return "Hold / Monitor"
    else:
        return "Normal"

replen_df["priority_flag"] = replen_df.apply(assign_priority_flag, axis=1)

# -----------------------------
# Select final columns
# -----------------------------
replen_support_table = replen_df[[
    "week_id",
    "week_start_date",
    "week_end_date",
    "week_type",
    "store_id",
    "store_name",
    "parent_sku_id",
    "child_sku_id",
    "lifecycle_type",
    "base_price",
    "sales_qty",
    "sales_value",
    "last_4w_avg_sales",
    "lead_time_weeks",
    "safety_stock_qty",
    "target_stock_qty",
    "on_hand_qty",
    "reorder_qty",
    "weeks_of_cover",
    "stock_status",
    "priority_flag"
]].copy()

# ============================================================
# 16. QUICK CHECKS
# ============================================================
print("Replenishment Support Table Sample")
display(replen_support_table.head(20))

print("Shape of replen_support_table:", replen_support_table.shape)

print("Stock status summary")
display(
    replen_support_table["stock_status"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "stock_status", "stock_status": "row_count"})
)

print("Priority flag summary")
display(
    replen_support_table["priority_flag"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "priority_flag", "priority_flag": "row_count"})
)

print("Reorder qty by store")
display(
    replen_support_table.groupby(["store_id", "store_name"])[["reorder_qty"]]
    .sum()
    .reset_index()
)

print("Average weeks of cover by stock status")
display(
    replen_support_table.groupby("stock_status")[["weeks_of_cover"]]
    .mean()
    .reset_index()
)

Replenishment Support Table Sample


,week_id,week_start_date,week_end_date,week_type,store_id,store_name,parent_sku_id,child_sku_id,lifecycle_type,base_price,...,sales_value,last_4w_avg_sales,lead_time_weeks,safety_stock_qty,target_stock_qty,on_hand_qty,reorder_qty,weeks_of_cover,stock_status,priority_flag
0,W001,2025-01-06,2025-01-12,spike,S001,Downtown Store,P001,P001_L,Evergreen,34,...,0,0.0,1,1,1,1,0,NaN,Understock,Normal
1,W002,2025-01-13,2025-01-19,spike,S001,Downtown Store,P001,P001_L,Evergreen,34,...,0,0.0,1,1,1,2,0,NaN,Overstock,Hold / Monitor
2,W003,2025-01-20,2025-01-26,normal,S001,Downtown Store,P001,P001_L,Evergreen,34,...,0,0.0,1,1,1,1,0,NaN,Understock,Normal
3,W004,2025-01-27,2025-02-02,normal,S001,Downtown Store,P001,P001_L,Evergreen,34,...,0,0.0,1,1,1,1,0,NaN,Understock,Normal
4,W005,2025-02-03,2025-02-09,slow,S001,Downtown Store,P001,P001_L,Evergreen,34,...,0,0.0,1,1,1,1,0,NaN,Understock,Normal
5,W006,2025-02-10,2025-02-16,normal,S001,Downtown Store,P001,P001_L,Evergreen,34,...,0,0.0,1,1,1,1,0,NaN,Understock,Normal
6,W007,2025-02-17,2025-02-23,normal,S001,Downtown Store,P001,P001_L,Evergreen,34,...,0,0.0,1,1,1,1,0,NaN,Understock,Normal
7,W008,2025-02-24,2025-03-02,normal,S001,Downtown Store,P001,P001_L,Evergreen,34,...,0,0.0,1,1,1,1,0,NaN,Understock,Normal
8,W009,2025-03-03,2025-03-09,slow,S001,Downtown Store,P001,P001_L,Evergreen,34,...,0,0.0,1,1,1,1,0,NaN,Understock,Normal
9,W010,2025-03-10,2025-03-16,slow,S001,Downtown Store,P001,P001_L,Evergreen,34,...,0,0.0,1,1,1,1,0,NaN,Understock,Normal


Shape of replen_support_table: (780, 21)
Stock status summary


,row_count,count
0,Understock,580
1,Overstock,115
2,Healthy,85


Priority flag summary


,row_count,count
0,Normal,614
1,Hold / Monitor,115
2,Reorder Now,51


Reorder qty by store


,store_id,store_name,reorder_qty
0,S001,Downtown Store,69
1,S002,Urban Store,51


Average weeks of cover by stock status


,stock_status,weeks_of_cover
0,Healthy,4.340941
1,Overstock,6.545238
2,Understock,2.473922


In [4]:
# ============================================================
# 17. EXPORT TABLES TO CSV
# ============================================================

import os

OUTPUT_FOLDER = "data_output"

# Create folder if not exists
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

# -----------------------------
# Save all tables
# -----------------------------
store_master.to_csv(f"{OUTPUT_FOLDER}/store_master.csv", index=False)

product_master_parent.to_csv(f"{OUTPUT_FOLDER}/product_master_parent.csv", index=False)

product_master_child.to_csv(f"{OUTPUT_FOLDER}/product_master_child.csv", index=False)

store_sku_master.to_csv(f"{OUTPUT_FOLDER}/store_sku_master.csv", index=False)

weekly_sales_fact.to_csv(f"{OUTPUT_FOLDER}/weekly_sales_fact.csv", index=False)

replen_support_table.to_csv(f"{OUTPUT_FOLDER}/replen_support_table.csv", index=False)

print("✅ All tables saved successfully to:", OUTPUT_FOLDER)

✅ All tables saved successfully to: data_output


In [4]:
# ============================================================
# PROMO CALENDAR TABLE
# Sales period: 2025-01-06 to 2025-06-30
# Promo period: 2025-01-06 to 2025-07-28
# ============================================================

import pandas as pd
import numpy as np
import random

# -----------------------------
# 1. Base weeks from weekly_sales_fact
# -----------------------------
DATA_PATH = "data_output"
weekly_sales_fact = pd.read_csv(f"{DATA_PATH}/weekly_sales_fact.csv")
promo_calendar = (
    weekly_sales_fact[["week_id", "week_start_date", "week_end_date"]]
    .drop_duplicates()
    .sort_values("week_start_date")
    .reset_index(drop=True)
)

promo_calendar["week_start_date"] = pd.to_datetime(promo_calendar["week_start_date"])
promo_calendar["week_end_date"] = pd.to_datetime(promo_calendar["week_end_date"])

# -----------------------------
# 2. Add extra weeks for July 2025
#    Existing sales ends at 2025-06-30
#    Need promo calendar until 2025-07-28
# -----------------------------
full_week_starts = pd.date_range(
    start="2025-01-06",
    end="2025-07-28",
    freq="W-MON"
)

existing_week_starts = set(promo_calendar["week_start_date"])
missing_week_starts = [d for d in full_week_starts if d not in existing_week_starts]

if len(missing_week_starts) > 0:
    max_existing_week_num = (
        promo_calendar["week_id"]
        .str.replace("W", "", regex=False)
        .astype(int)
        .max()
    )

    extra_rows = []
    next_week_num = max_existing_week_num + 1

    for week_start in missing_week_starts:
        extra_rows.append({
            "week_id": f"W{str(next_week_num).zfill(3)}",
            "week_start_date": week_start,
            "week_end_date": week_start + pd.Timedelta(days=6)
        })
        next_week_num += 1

    promo_calendar_extra = pd.DataFrame(extra_rows)

    promo_calendar = pd.concat(
        [promo_calendar, promo_calendar_extra],
        ignore_index=True
    )

promo_calendar = promo_calendar.sort_values("week_start_date").reset_index(drop=True)

# -----------------------------
# 3. Add promo fields
# -----------------------------
promo_calendar["promo_type"] = None
promo_calendar["promo_name"] = None
promo_calendar["is_promo"] = 0

# -----------------------------
# 4. Fixed promo schedule
#    Keep this deterministic and explainable
# -----------------------------
promo_plan = {
    "2025-01-27": ("Discount", "Month End Push"),
    "2025-02-24": ("Bundle", "Payday Bundle"),
    "2025-03-31": ("Campaign", "Quarter End Campaign"),
    "2025-04-28": ("Discount", "Summer Kickoff"),
    "2025-05-26": ("Bundle", "Payday Bundle"),
    "2025-06-30": ("Campaign", "Mid Year Sale"),
    "2025-07-28": ("Discount", "End of July Clearance")
}

for week_start_str, promo_info in promo_plan.items():
    promo_type, promo_name = promo_info
    mask = promo_calendar["week_start_date"] == pd.Timestamp(week_start_str)

    promo_calendar.loc[mask, "promo_type"] = promo_type
    promo_calendar.loc[mask, "promo_name"] = promo_name
    promo_calendar.loc[mask, "is_promo"] = 1

# -----------------------------
# 5. Optional helper field
#    Distinguish weeks that exist in sales vs future promo-only weeks
# -----------------------------
promo_calendar["in_sales_period"] = np.where(
    promo_calendar["week_start_date"] <= pd.Timestamp("2025-06-30"),
    1,
    0
)

# -----------------------------
# 6. Check
# -----------------------------
display(promo_calendar)
print(promo_calendar.shape)

,week_id,week_start_date,week_end_date,promo_type,promo_name,is_promo,in_sales_period
0,W001,2025-01-06,2025-01-12,None,None,0,1
1,W002,2025-01-13,2025-01-19,None,None,0,1
2,W003,2025-01-20,2025-01-26,None,None,0,1
3,W004,2025-01-27,2025-02-02,Discount,Month End Push,1,1
4,W005,2025-02-03,2025-02-09,None,None,0,1
5,W006,2025-02-10,2025-02-16,None,None,0,1
6,W007,2025-02-17,2025-02-23,None,None,0,1
7,W008,2025-02-24,2025-03-02,Bundle,Payday Bundle,1,1
8,W009,2025-03-03,2025-03-09,None,None,0,1
9,W010,2025-03-10,2025-03-16,None,None,0,1


(30, 7)


In [6]:
OUTPUT_FOLDER = "data_output"
promo_calendar.to_csv(f"{OUTPUT_FOLDER}/promo_calendar.csv", index=False)